# Checking for duplicates in database

This notebook analyses if there are any duplicates inside the tables of database.

In [311]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from database.scripts.config import load_config
from database.scripts.connect import connect

# ignore warings
import warnings
warnings.filterwarnings("ignore")

# plot defaults
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# load db config data
config = load_config("database/scripts/database.ini")
# connect to posgres db
conn = connect(config)

# execute sql and return results as DataFrame
def q(sql, params=None):
    return pd.read_sql(sql, conn, params=params)

Connected to the PostgreSQL server.


## Entries in work

In [312]:
q("""
    SELECT count(*)
    FROM openalex.work;
""")

,count
0,3860


## Duplicates in work

Checking for duplicates on entries in doi column.

In [313]:
work_doi = q("""
    SELECT doi, count(*) AS n, array_agg(id) AS ids
    FROM openalex.work
    WHERE doi IS NOT NULL
    GROUP BY doi
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{work_doi["n"].sum()} duplicates found")

work_doi

0 duplicates found


,doi,n,ids


## Handle duplicate DOIs in work table

When inserting works from OpenAlex data duplicates on DOI can appear. To address this issue the one with the newer publication year will be kept.

#### <u>Insert logic</u>

- If an existing work with same DOI is **newer or equally recent** -> skip the insert.
- If an existing work with same DOI is **older** -> delete it, then insert the new one.
- IF **no existing work** has this DOI -> insert it directly.


Investigations showed that whenever inserting new works a duplicate DOI can occur. The existing work in the database was always the **older** publication. Means, the publication year was less then or equal to the current one.

#### <u>Pseudocode within insertion process</u>
```python
existing_publication_year = get_work_publication_year(doi)
if existing_publication_year is not None:
    if existing_publication_year >= current_publication_year:
        return  # keep work with existing DOI
    delete_work(doi) # delete existing work
insert_work() # insert current work
```

Checking for duplicates on entries in normalized title column.

In [314]:
work_title = q("""
    SELECT lower(trim(title)) AS title_norm, count(*) AS n, array_agg(id) AS ids
    FROM openalex.work
    GROUP BY title_norm
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{work_title["n"].sum()} duplicates found")

work_title

0 duplicates found


,title_norm,n,ids


## Handle duplicate normalized titles in work table

When inserting works from OpenAlex data, duplicates on normalized title can appear even when DOIs differ. To resolve this issue, the exisiting work and the current work are compared field by field, in priority oder: `publication_date`,`authors_count`,`updated_date`. The first field where one side strictly wins decides the outcome; ties on a field fall thorugh to the next one. The field `authors_count` is checked, because same works with same titles can differ in `authors_count`. The work with more authors will be kept.

#### <u>Insert logic</u>

- If the current work has **no publication_date** -> skip the insert (existing work is kept).
- Otherwise, compare `publication_date`, then `authors_count`, then `updated_date`, in order:
  - First field where **current > existing** -> delete the existing work, keep the current one.
  - First field where **current < existing** -> keep the existing work, skip the insert.
  - Equal on a field -> move to the next field in the list.
- If **all three fields are tied** -> keep the existing work, skip the insert.
- If **no existing work** has this normalized title -> insert it directly.

#### <u>Pseudocode within insertion process</u>
```python
if exists_normalized_title(title):
    existing_work = get_work_by_title(title)
    if current_publication_date is None:
        return # skip: current work misses publication_date
    for field in ["publication_date", "authors_count", "updated_date"]:
        if current[field] > existing[field]:
            delete_work(existing_work.id)
            break
        if current[field] < existing[field]
            return # keep existing work
    else:
        return # all fields tied -> keep existing work
insert_work() # no title duplicate or current work won on fields
```

## Entries in author

In [315]:
q("""
    SELECT count(*)
    FROM openalex.author;
""")

,count
0,14572


## Duplicates in author

Checking for duplicates on enties in openalex_id column.

In [316]:
author_openalex_id = q("""
    SELECT openalex_id, count(*) AS n, array_agg(id) AS ids
    FROM openalex.author
    WHERE openalex_id IS NOT NULL
    GROUP BY openalex_id
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{author_openalex_id["n"].sum()} duplicates found")

author_openalex_id

0 duplicates found


,openalex_id,n,ids


Checking for duplicates on entries in orcid column.

In [317]:
author_orcid = q("""
    SELECT orcid, count(*) AS n, array_agg(id) AS ids
    FROM openalex.author
    WHERE orcid IS NOT NULL
    GROUP BY orcid
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{author_orcid["n"].sum()} duplicates found")

author_orcid

0 duplicates found


,orcid,n,ids


Check for duplicates on entries in display_name column.

In [318]:
author_display_name = q("""
    SELECT lower(trim(display_name)) AS display_name_norm, count(*) AS n, array_agg(id) AS ids
    FROM openalex.author
    GROUP BY display_name_norm
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{author_display_name["n"].sum()} duplicates found")

author_display_name

522 duplicates found


,display_name_norm,n,ids
0,wei wang,7,"[3233, 10064, 10770, 9885, 6250, 7316, 7297]"
1,yu li,6,"[5028, 12156, 5161, 4863, 6070, 4508]"
2,lei chen,4,"[7379, 8207, 374, 4039]"
3,qi chen,4,"[5323, 8184, 2171, 7709]"
4,yang liu,4,"[5185, 6662, 3483, 8494]"
...,...,...,...
229,hui yu,2,"[9684, 2853]"
230,hui zhang,2,"[2692, 3439]"
231,h. zhang,2,"[2901, 1776]"
232,ida bagus nyoman pascima,2,"[1672, 14031]"


These are different authors with same names. Can't tell if they are the same person, because openalex_id differs.

Check for duplicates on entries in all columns vaules combined.

In [319]:
author_duplicates = q("""
    SELECT openalex_id, display_name, orcid, COUNT(*) AS n, array_agg(id) AS ids
    FROM openalex.author
    GROUP BY openalex_id, display_name, orcid
    HAVING COUNT(*) > 1;
""")

print(f"{author_duplicates["n"].sum()} duplicates found")

author_duplicates

0 duplicates found


,openalex_id,display_name,orcid,n,ids


## Handle duplicates in author table

When inserting auhtors from OpenAlex authorships data, the same person can be encountered multiple times with different levels of identifying information (sometimes with an `openalex_id` or `orcid`, sometimes with neither). To avoid creating duplicate rows, authors are resolved through a tiered lookup before any inserts happens.

#### <u>Insert logic</u>

1. **If `openalex_id` or `orcid` is present** -> look up an existing row where `openalex_id` matches **or** `orcid` matches (whichever of the two is not null on the incoming record).
    - **Match found** -> update that row's `openalex_id`, `display_name` and `orcid` using `COALESCE`, so only values that are not null overwrite the existing ones. After update, save author `id`.
    - **No match** -> insert new author and save `id`.
2. **If neither `openalex_id` nor `orcid` is present** -> try to resolve by `display_name` **and** institution overlap (`get_author_id_by_display_name_and_institutions`).
    - **Match found** -> Save author `id`.
    - **No match** -> fall back to `display_name` only (`get_author_id_by_display_name`), ignoring institution relations.
        - **Match found** -> Save author `id`.
        - **No match** -> insert a new author and save `id`.
3. In every branch, the resulting `(author_id, institution_ids)` pair is stored in a list. This list is returned at the end and used for insertion process in `work_author_institution` table.

#### <u>Pseudocode within insertion process</u>
```python
# ...
for openalex_id, display_name, orcid, institution_ids in authors:
    if openalex_id or orcid:
        existing = get_author_by_openalex_id_or_orcid(openalex_id, orcid)
        if existing:
            author_id = update_author_coalesced(existing.id, openalex_id, display_name, orcid)
        else:
            auhtor_id = insert_author(openalex_id, display_name, orcid)
    else: 
        author_id = get_author_id_by_display_name_and_institutions(current_display_name, current_institution_ids)
        if author_id is None:
            author_id = get_author_id_by_display_name(current_display_name)
        if author_id is None:
            author_id = insert_author(openalex_id, display_name, orcid) 
    authors_with_institution_ids.append((author_id, current_institution_ids)) # will be returned
```

## Entries in institution

In [320]:
q("""
    SELECT count(*)
    FROM openalex.institution;
""")

,count
0,3715


## Duplicates in institution

Checking for duplicates on entries in ror column.

In [321]:
institution_ror = q("""
    SELECT lower(trim(ror)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.institution
    WHERE ror IS NOT NULL
    GROUP BY ror
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{institution_ror["n"].sum()} duplicates found")

institution_ror

0 duplicates found


,lower,n,ids


Checking duplicates on entries in display_name column.

In [322]:
institution_display_name = q("""
    SELECT lower(trim(display_name)) AS display_name_norm, count(*) AS n, array_agg(id) AS ids
    FROM openalex.institution
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{institution_display_name["n"].sum()} duplicates found")

institution_display_name

49 duplicates found


,display_name_norm,n,ids
0,institut de recherche pour le développement,3,"[I4210108561, I1306264927, I4210166444]"
1,arab open university,2,"[I4210090878, I4210139873]"
2,bioinformatics institute,2,"[I4210148498, I4210137637]"
3,carter center,2,"[I1292524976, I4210096273]"
4,cisco systems (united states),2,"[I135428043, I4210129566]"
5,cracow university of technology,2,"[I24881138, I4210092770]"
6,institute of mechanics,2,"[I4210148896, I4210157653]"
7,institute of philosophy,2,"[I4210104258, I4210088627]"
8,institute of physics,2,"[I4210159876, I4210086947]"
9,joint research centre,2,"[I4210118689, I4210162697]"


These are same institutions with different in countries.

## Handle duplicates in institution table

Duplicates are resolved through **`UNIQUE`** modifier on field **`ror`** and checking if institution id is alread present in databse (**`ON CONFLICT (ror) DO NOTHING`**).


Insertion logic:

- If institution `id` **IS** present in table -> skipp instition insert
- If institution `id` **IS NOT** present in table -> insert institution

#### <u>Pseudocode within insertion process</u>
```python
if not existing_institution(id):
    insert_institution()
```


## Entries in funder

In [323]:
q("""
    SELECT count(*)
    FROM openalex.funder;
""")

,count
0,892


## Duplicates in funder

Checking for duplicates on entries in ror column.

In [324]:
funder_ror = q("""
    SELECT ror, COUNT(*) AS n, array_agg(id) AS funder_ids
    FROM openalex.funder
    WHERE ror IS NOT NULL
    GROUP BY ror
    HAVING COUNT(*) > 1
    ORDER BY n DESC;
""")

print(f"{funder_ror["n"].sum()} duplicates found")

funder_ror

0 duplicates found


,ror,n,funder_ids


Checking for duplicates on entries in display_name column.

In [325]:
funder_display_name = q("""
    SELECT lower(trim(display_name)) as display_name_norm, count(*) AS n, array_agg(id) AS ids
    FROM openalex.funder
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{funder_display_name["n"].sum()} duplicates found")

funder_display_name

0 duplicates found


,display_name_norm,n,ids


## Handle duplicates in funder table

When inserting funders from OpenAlex data, a funder may already exist in the table because its OpenAlex `id` was alread inserted. To avoid duplicate rows or unnecessary re-inserts, funders are resolved by unique identifier `ror` before attempting an insert, with an `ON CONFLICT` clause as a secondary safeguard on `id`.

#### <u>Insert logic</u>

1. **Look up an existing funder by `ror`** (`find_funder_by_ror`).
    - **Match found** -> Save that funder's existing `id`. No insert or update is performed.
    - **No match** -> attempt to insert the funder as new, using its OpenAlex `id`.
2. **On insert, `ON CONFLICT (id) DO UPDATE` acts as a fallback** for the case where the `ror` lookup found nothing but the `id` already exists in the table.
    - `ror` is set to `CASCADE(openalex.funder.ror, EXCLUDED.ror)`. That means, if the existing `ror` is `NULL` and the current value of `ror` is `NOT NULL` the ror value will be updated.
    - `display_name` is **not** updated on conflict - the originally inserted `display_name` is always kept.
3. The resulting `funder_id` either saved from matching or from inserting is append to `funder_ids` list.<br>

*The returned `funder_ids` will be used to insert relations between funders and works.*


#### <u>Pseudocode within insertion process</u>
```python
for funder in funders:
    existing_id = find_funder_by_ror(funder["ror"])
    if existing_id:
        funder_id = existing_id
    else:
        funder_id = insert_funder_on_conflict_update_ror(funder["id"], funder["display_name"], funder["ror"])
    funder_ids.append(funder_id) # will be returned
```


## Entries in source

In [326]:
q("""
    SELECT count(*)
    FROM openalex.source;
""")

,count
0,2091


## Duplicates in source

Checking for duplicates on entries in issn_l column.

In [327]:
source_issn_l = q("""
    SELECT lower(trim(issn_l)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.source
    WHERE issn_l IS NOT NULL
    GROUP BY issn_l
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{source_issn_l["n"].sum()} duplicates found")

source_issn_l

0 duplicates found


,lower,n,ids


Checking for duplicates on entries in display_name column.

In [328]:
source_display_name = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.source
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{source_display_name["n"].sum()} duplicates found")

source_display_name

27 duplicates found


,lower,n,ids
0,spire - sciences po institutional repository,5,"[S4406922454, S4406922398, S4406922461, S44069..."
1,geodesy and cartography,2,"[S12104227, S4210169005]"
2,infoscience (ecole polytechnique fédérale de l...,2,"[S4306400487, S4306400488]"
3,international journal of computer science and ...,2,"[S154051528, S4390963318]"
4,international journal of educational development,2,"[S5407050336, S20152851]"
5,journal of geophysical research atmospheres,2,"[S207178839, S4210205282]"
6,london school of economics and political scien...,2,"[S4306401594, S4306401593]"
7,pub – publications at bielefeld university (bi...,2,"[S4306401670, S4306401671]"
8,quaestiones geographicae,2,"[S157707066, S4210171875]"
9,repository@nottingham (university of nottingham),2,"[S4306402481, S4306402483]"


## Entries in locations

In [329]:
q("""
    SELECT count(*)
    FROM openalex.locations;
""")

,count
0,7195


## Duplicates in locations

Checking for duplicates on entries in pdf_url column.

In [330]:
locations_pdf_url = q("""
    SELECT lower(trim(pdf_url)) AS pdf_url_norm, count(*) AS n, array_agg(id) AS ids
    FROM openalex.locations
    WHERE pdf_url IS NOT NULL
    GROUP BY pdf_url
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{locations_pdf_url["n"].sum()} duplicates found")

locations_pdf_url

0 duplicates found


,pdf_url_norm,n,ids


Checking for duplicates on entries in landing_page_url column.

In [331]:
locations_landing_page_url = q("""
    SELECT lower(trim(landing_page_url)) AS landing_page_url_norm, count(*) AS n, array_agg(id) AS ids
    FROM openalex.locations
    WHERE landing_page_url IS NOT NULL
    GROUP BY landing_page_url
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{locations_landing_page_url["n"].sum()} duplicates found")

locations_landing_page_url

8 duplicates found


,landing_page_url_norm,n,ids
0,https://orcid.org/0000-0003-4640-824x,2,"[pmh:oai:researchonline.ljmu.ac.uk:28990, pmh:..."
1,https://orcid.org/0009-0008-2092-7029,2,"[oai:ray.yorksj.ac.uk:11412, oai:ray.yorksj.ac..."
2,https://research.birmingham.ac.uk/en/publicati...,2,[oai:pure.atira.dk:openaire_cris_publications/...
3,https://researchonline.lshtm.ac.uk/view/creato...,2,"[oai:researchonline.lshtm.ac.uk:4649421, oai:r..."


## Entries in keyword

In [332]:
q("""
    SELECT count(*)
    FROM openalex.keyword;
""")

,count
0,5159


## Duplicates in keyword

In [333]:
keyword = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.keyword
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{keyword["n"].sum()} duplicates found")

keyword

0 duplicates found


,lower,n,ids


## Entries in domain

In [334]:
q("""
    SELECT count(*)
    FROM openalex.domain;
""")

,count
0,4


## Duplicates in domain

Each domain has it's own OpenAlex identifier. No duplication check needed.

In [335]:
domain = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.domain
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{domain["n"].sum()} duplicates found")

domain

0 duplicates found


,lower,n,ids


## Entries in field

In [336]:
q("""
    SELECT count(*)
    FROM openalex.field;
""")

,count
0,26


## Duplicates in field

Each field has it's own OpenAlex identifier. No duplication check needed.

In [337]:
field = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.field
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{field["n"].sum()} duplicates found")

field

0 duplicates found


,lower,n,ids


## Entries in subfield

In [338]:
q("""
    SELECT count(*)
    FROM openalex.subfield;
""")

,count
0,202


## Duplicates in subfield

Checking for duplicates on entries in display_name column. Allthough this should have it's own identifier from OpenAlex.

In [339]:
subfield = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.subfield
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{subfield["n"].sum()} duplicates found")

subfield

8 duplicates found


,lower,n,ids
0,industrial and manufacturing engineering,2,"[2209, 2311]"
1,genetics,2,"[1311, 2716]"
2,pharmacology,2,"[2736, 3004]"
3,neurology,2,"[2808, 2728]"


These are same subfields but on different fields.

## Entries in topic

In [340]:
q("""
    SELECT count(*)
    FROM openalex.topic;
""")

,count
0,1611


## Duplicates in topic

Each topic has it's own OpenAlex identifier. No duplication check needed.

In [341]:
topic = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.topic
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{topic["n"].sum()} duplicates found")

topic

0 duplicates found


,lower,n,ids


In [342]:
conn.close()